# Scoring ranges analysis

Analysis of `scoring_ranges_aggregate.csv` — semantic similarity and perplexity distributions per strategy at `--n = 50` across 117 seeds from the volume sweep (16,962 variants total).

**Goals:**
1. Produce descriptive distributions (median, IQR, p95) per strategy for both metrics
2. Characterize how distributions vary by utterance length
3. Quantify whether the README's current filter thresholds (0.55 with expansion, 0.70 without) separate strategies as claimed
4. Surface the extreme-perplexity tail and its cause
5. Test the design claim that the four strategies produce textually complementary output

**Caveats baked into the study design (all documented in the writeup):**
- All runs used temperature 0.9. Distributions here are temperature-conditional.
- GPT-2 perplexity is length-sensitive; cross-strategy perplexity comparisons are partially confounded by variant length.
- We're describing allo's self-reported scores, not external validation.
- Exact-match deduplication happens before the CSV is written, so cross-strategy overlap is measured on what survives dedup, not on raw output.

## Setup

In [ ]:
import json
from collections import Counter
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ─── Configuration ───────────────────────────────────────────
AGGREGATE_PATH = Path("../results/scoring_ranges/scoring_ranges_aggregate.csv")
RESULTS_DIR = Path("../results/scoring_ranges")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

STRATEGY_ORDER = ["llm_paraphrase", "constrained", "mlm_substitution", "expansion"]
STRATEGY_COLORS = {
    "llm_paraphrase": "#2E86AB",
    "constrained": "#A23B72",
    "mlm_substitution": "#E07A5F",
    "expansion": "#81B29A",
}
LENGTH_COLORS = {"short": "#D62828", "medium": "#F77F00", "long": "#003049"}

In [ ]:
df = pd.read_csv(AGGREGATE_PATH)

print(f"Loaded {len(df)} variants from {AGGREGATE_PATH.name}")
print(f"Seeds: {df['seed_id'].nunique()}")
print(f"Strategies: {sorted(df['strategy_family'].unique())}")

print("\nVariants per strategy:")
print(df['strategy_family'].value_counts())

print("\nVariants per length:")
print(df['utterance_length'].value_counts())

# Compute once and reuse: the visual cap for perplexity plots
PERPLEXITY_P99 = np.percentile(df['perplexity'], 99)
PERPLEXITY_P01 = np.percentile(df['perplexity'], 1)
print(f"\nPerplexity p99 = {PERPLEXITY_P99:.1f} (used as visual cap)")
print(f"Perplexity p01 = {PERPLEXITY_P01:.1f} (used as visual floor)")

---

## Section 1: Headline distributions

Per-strategy descriptive statistics for both metrics. These are the numbers that will replace the qualitative guidance in the README's "Interpreting scores across strategies" section.

**Why median, IQR, p95 instead of mean and stdev:** The perplexity distribution has an extreme right tail (max values over 100,000 for some strategies). Means are pulled substantially upward by these outliers and don't describe typical variants well. Medians and percentiles are stable against outliers. Section 4 documents where the extreme tail comes from.

In [ ]:
# Descriptive statistics table — the headline deliverable of this study
stats = []
for strat in STRATEGY_ORDER:
    for metric in ["semantic_similarity", "perplexity"]:
        s = df[df["strategy_family"] == strat][metric]
        stats.append({
            "strategy": strat,
            "metric": metric,
            "n": len(s),
            "min": round(s.min(), 2),
            "p05": round(np.percentile(s, 5), 2),
            "p25": round(np.percentile(s, 25), 2),
            "median": round(s.median(), 2),
            "p75": round(np.percentile(s, 75), 2),
            "p95": round(np.percentile(s, 95), 2),
            "max": round(s.max(), 1),
        })
stats_df = pd.DataFrame(stats)
stats_df.to_csv(RESULTS_DIR / "scoring_summary.csv", index=False)
stats_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Similarity — linear scale
ax = axes[0]
data_sim = [df[df["strategy_family"] == s]["semantic_similarity"].values for s in STRATEGY_ORDER]
bp1 = ax.boxplot(data_sim, tick_labels=STRATEGY_ORDER, showfliers=False,
                 patch_artist=True, widths=0.6)
for patch, strat in zip(bp1["boxes"], STRATEGY_ORDER):
    patch.set_facecolor(STRATEGY_COLORS[strat])
    patch.set_alpha(0.6)
ax.set_ylabel("Semantic similarity to seed", fontsize=11)
ax.set_title("Semantic similarity distribution per strategy", fontsize=12)
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(-0.05, 1.05)

# Perplexity — log scale, capped at p99 so outliers don't dominate
ax = axes[1]
data_ppl = [df[df["strategy_family"] == s]["perplexity"].values for s in STRATEGY_ORDER]
bp2 = ax.boxplot(data_ppl, tick_labels=STRATEGY_ORDER, showfliers=False,
                 patch_artist=True, widths=0.6)
for patch, strat in zip(bp2["boxes"], STRATEGY_ORDER):
    patch.set_facecolor(STRATEGY_COLORS[strat])
    patch.set_alpha(0.6)
ax.set_yscale("log")
ax.set_ylabel("GPT-2 perplexity (log scale)", fontsize=11)
ax.set_title("Perplexity distribution per strategy (log scale, capped at p99)", fontsize=12)
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(PERPLEXITY_P01 * 0.8, PERPLEXITY_P99 * 1.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "headline_distributions.png", dpi=120)
plt.show()

**What this shows:**

- **Similarity medians** separate the four strategies cleanly: constrained 0.89, mlm_substitution 0.87, llm_paraphrase 0.83, expansion 0.64. The three paraphrase-class strategies sit in a tight band around 0.83–0.89; expansion sits well below at 0.64, consistent with its design intent of producing adjacent-intent variants rather than paraphrases.
- **Perplexity medians** are all in the 99–154 range — within the 50–200 "typical natural English" band that `evaluate.py`'s docstring names. Constrained is lowest (99), MLM and expansion highest (153, 154), llm_paraphrase in between (123).
- **Perplexity p95 values are 661–1,188 across strategies** — an order of magnitude above the median. This is where the distribution starts to include variants that GPT-2 genuinely finds unusual. Section 4 documents what drives the right tail.

---

## Section 2: Distributions by utterance length

The volume sweep showed that each strategy behaves differently on short vs. long seeds. We check whether the scoring distributions show the same length-gating.

In [ ]:
# Similarity median by (strategy, length)
sim_by_length = df.pivot_table(
    index="strategy_family",
    columns="utterance_length",
    values="semantic_similarity",
    aggfunc="median",
).round(3)[["short", "medium", "long"]]
print("Median similarity by (strategy, length):")
print(sim_by_length)

# IQR by (strategy, length) — narrow IQRs indicate tight clustering
print("\nSimilarity IQR (p75 - p25) by (strategy, length):")
iqr_rows = []
for strat in STRATEGY_ORDER:
    for length in ["short", "medium", "long"]:
        sub = df[(df["strategy_family"] == strat) & (df["utterance_length"] == length)]["semantic_similarity"]
        iqr_rows.append({
            "strategy": strat,
            "length": length,
            "p25": round(np.percentile(sub, 25), 3),
            "p75": round(np.percentile(sub, 75), 3),
            "iqr": round(np.percentile(sub, 75) - np.percentile(sub, 25), 3),
        })
iqr_df = pd.DataFrame(iqr_rows)
print(iqr_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, strat in zip(axes, STRATEGY_ORDER):
    sub = df[df["strategy_family"] == strat]
    data = [sub[sub["utterance_length"] == L]["semantic_similarity"].values
            for L in ["short", "medium", "long"]]
    bp = ax.boxplot(data, tick_labels=["short", "medium", "long"],
                    showfliers=False, patch_artist=True, widths=0.6)
    for patch, L in zip(bp["boxes"], ["short", "medium", "long"]):
        patch.set_facecolor(LENGTH_COLORS[L])
        patch.set_alpha(0.6)
    ax.set_title(strat, fontsize=11)
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_ylim(-0.05, 1.05)
axes[0].set_ylabel("Semantic similarity", fontsize=11)
plt.suptitle("Semantic similarity by utterance length, per strategy", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "similarity_by_length.png", dpi=120)
plt.show()

In [ ]:
# Perplexity by (strategy, length) — log-scale plot
ppl_by_length = df.pivot_table(
    index="strategy_family",
    columns="utterance_length",
    values="perplexity",
    aggfunc="median",
).round(1)[["short", "medium", "long"]]
print("Median perplexity by (strategy, length):")
print(ppl_by_length)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, strat in zip(axes, STRATEGY_ORDER):
    sub = df[df["strategy_family"] == strat]
    data = [sub[sub["utterance_length"] == L]["perplexity"].values
            for L in ["short", "medium", "long"]]
    bp = ax.boxplot(data, tick_labels=["short", "medium", "long"],
                    showfliers=False, patch_artist=True, widths=0.6)
    for patch, L in zip(bp["boxes"], ["short", "medium", "long"]):
        patch.set_facecolor(LENGTH_COLORS[L])
        patch.set_alpha(0.6)
    ax.set_yscale("log")
    ax.set_title(strat, fontsize=11)
    ax.grid(True, alpha=0.3, axis="y")
    ax.set_ylim(PERPLEXITY_P01 * 0.8, PERPLEXITY_P99 * 1.3)
axes[0].set_ylabel("GPT-2 perplexity (log)", fontsize=11)
plt.suptitle("Perplexity by utterance length, per strategy (log scale, capped at p99)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "perplexity_by_length.png", dpi=120)
plt.show()

**Two consistent patterns in the length data:**

1. **Median similarity rises with length** across every strategy. For MLM the effect is largest (short 0.81 → long 0.95); for constrained and llm_paraphrase it's subtler (0.86 → 0.92 and 0.81 → 0.86). Short seeds offer less lexical material to substitute or restructure, so each variation changes a larger fraction of the utterance — and embedding similarity drops accordingly. This is a mechanical consequence of utterance length, not a quality signal.

2. **Median perplexity falls with length** across every strategy. For mlm_substitution: 211 (short) → 158 (medium) → 97 (long). For llm_paraphrase: 149 → 117 → 92. This is the known GPT-2 length-sensitivity — shorter sequences have less context for the model to condition on, and the average per-token negative log-likelihood runs higher.

Both patterns are worth documenting but neither is a strategy-quality finding. They're properties of the metrics, not the strategies. This matters for users who might naively threshold these values: a `min_similarity=0.80` filter, for example, will disproportionately remove short-seed variants regardless of which strategy generated them.

---

## Section 3: Do the README's similarity thresholds actually separate strategies?

The README currently recommends `--filter-min-similarity 0.55` when expansion is enabled, and `0.70` otherwise. These numbers were set by intuition. Now that we have distributions, we can check whether they actually achieve what's intended.

In [ ]:
# What fraction of each strategy falls below various similarity cutoffs?
thresholds = [0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
rows = []
for strat in STRATEGY_ORDER:
    sub = df[df["strategy_family"] == strat]["semantic_similarity"]
    row = {"strategy": strat, "n": len(sub)}
    for t in thresholds:
        row[f"<{t:.2f}"] = f"{(sub < t).mean():.1%}"
    rows.append(row)
pd.DataFrame(rows)

In [ ]:
# Strategy separation visualisation
fig, ax = plt.subplots(figsize=(10, 5.5))
bins = np.linspace(0, 1, 41)
for strat in STRATEGY_ORDER:
    sub = df[df["strategy_family"] == strat]["semantic_similarity"]
    ax.hist(sub, bins=bins, alpha=0.5, label=strat, color=STRATEGY_COLORS[strat], density=True)

ax.axvline(0.55, color="black", linestyle=":", alpha=0.7,
           label="README's 0.55 floor (with expansion)")
ax.axvline(0.70, color="gray", linestyle=":", alpha=0.5,
           label="README's 0.70 floor (without expansion)")
ax.set_xlabel("Semantic similarity", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Strategy separation in similarity space", fontsize=12)
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "strategy_separation.png", dpi=120)
plt.show()

**What the threshold analysis shows:**

- **`--filter-min-similarity 0.55` (with expansion enabled):** Removes 29% of expansion variants while preserving 99.5% of constrained, 95.5% of llm_paraphrase, and 95.0% of mlm_substitution. The floor is working roughly as intended — it removes the most-drifted expansion variants while preserving essentially all paraphrase-class output. But note that 29% of expansion is a substantial cut; many of those variants are by-design adjacent rather than synonymous.

- **`--filter-min-similarity 0.70` (without expansion):** Removes 17% of llm_paraphrase, 16% of mlm_substitution, and 7% of constrained. This is aggressive — nearly one in six paraphrase-class variants falls below the threshold. Users using this setting should expect to discard meaningful volume.

- **The histogram shows clean modal separation.** Expansion peaks around 0.60–0.70; the three paraphrase-class strategies peak between 0.85–0.95. The 0.55 threshold sits squarely in the valley between these two modes, which is why it works reasonably as a separator. The 0.70 threshold sits inside the paraphrase-class mass, which is why it discards more paraphrase content.

Conclusion: the README's recommended thresholds are defensible but not precisely calibrated. A more aggressive filter like `0.65` with expansion would remove ~51% of expansion while still preserving ~88% of paraphrase-class variants — worth naming as an option for users who want to discard more of expansion's fringe.

---

## Section 4: The extreme perplexity tail is driven by two seeds

Maximum perplexity values range from 54,385 to 613,649 across strategies. These are dramatic outliers — not a gradual tail. This section documents what drives them.

In [ ]:
# How many variants are in the extreme tail?
for threshold in [500, 1000, 5000, 10000]:
    n_above = (df["perplexity"] > threshold).sum()
    pct = n_above / len(df) * 100
    print(f"Variants with perplexity > {threshold:>5}: {n_above:>4} ({pct:.2f}%)")

In [ ]:
# Which seeds are responsible for the extreme tail?
extreme = df[df["perplexity"] > 10000]
seed_counts = extreme.groupby("seed").size().sort_values(ascending=False)
print("Seeds producing ppl > 10,000 variants:")
print(seed_counts)
print(f"\nTotal extreme variants: {len(extreme)}")
print(f"Top 2 seeds account for: {seed_counts.head(2).sum()} / {len(extreme)} "
      f"= {seed_counts.head(2).sum() / len(extreme):.0%}")

In [ ]:
# For the two outlier seeds: what are their variants, and what do their scores look like?
outlier_seeds = ["call call mom", "play something uh relaxing"]

for s in outlier_seeds:
    sub = df[df["seed"] == s]
    print(f"\n{'=' * 60}")
    print(f"Seed: '{s}' ({len(sub)} variants)")
    print(f"{'=' * 60}")
    print(f"Median perplexity: {sub['perplexity'].median():.1f}")
    print(f"Mean perplexity:   {sub['perplexity'].mean():.1f}")
    print(f"Max perplexity:    {sub['perplexity'].max():.1f}")
    print(f"\nFor comparison, 'turn off the lights' has:")
    normal = df[df["seed"] == "turn off the lights"]
    print(f"  Median perplexity: {normal['perplexity'].median():.1f}")
    print(f"  Max perplexity:    {normal['perplexity'].max():.1f}")

In [ ]:
# Does removing the two outlier seeds meaningfully tighten the p95 perplexity?
print("Per-strategy p95 perplexity: with vs without the 2 outlier seeds")
print()
for strat in STRATEGY_ORDER:
    sub_with = df[df["strategy_family"] == strat]["perplexity"]
    sub_without = df[
        (df["strategy_family"] == strat) & (~df["seed"].isin(outlier_seeds))
    ]["perplexity"]
    p95_with = np.percentile(sub_with, 95)
    p95_without = np.percentile(sub_without, 95)
    print(f"  {strat:20s}  with outliers: {p95_with:>7.1f}   "
          f"without: {p95_without:>7.1f}   "
          f"delta: {p95_with - p95_without:>6.1f}")

**The two outlier seeds explain 85% of the extreme tail.**

- **`call call mom`** (21 extreme variants) contains a repeated token. Variants that normalize the repetition, such as `constrained:abbreviated_spoken` producing "call mom" (perplexity 61,611), score extraordinarily high despite being semantically correct (similarity 0.94). The cause is GPT-2's length sensitivity: 2-word utterances have almost no conditioning context, and the model's per-token loss runs very high. This is a known property of GPT-2 perplexity on short strings, documented in `evaluate.py`'s docstring. It is not a quality signal about the variant.

- **`play something uh relaxing`** (26 extreme variants) contains a filled-pause token (`uh`). MLM substitution tends to preserve the `uh` while changing another content word, producing outputs like "play your uh relaxing" (perplexity 87,759). GPT-2 treats `uh` in isolation as extremely unlikely, and this inflates the perplexity regardless of what else the variant does.

Removing these two seeds tightens the per-strategy p95 perplexity modestly — 56–150 units across strategies — but the story is not that the p95 changes dramatically. The story is that **the max values are misleading as distribution descriptors**. The medians and IQRs in Section 1 are the trustworthy numbers.

**Implication for users:** perplexity is a length-sensitive and content-sensitive metric. Users should not set `--filter-max-perplexity` aggressively without first inspecting which variants are being removed. A threshold of 500 would remove 11% of all variants, and those variants cluster around short and disfluent seeds rather than around genuinely low-quality output.

---

## Section 5 (stretch): cross-strategy overlap

The design claim is that the four strategies produce textually complementary output. If the same (seed, utterance) pair were produced by multiple strategies, the strategies would be redundant.

In [ ]:
# How many (seed, utterance) pairs appear in more than one strategy?
per_utt = df.groupby(["seed_id", "utterance"])["strategy_family"].nunique()

print(f"Total unique (seed, utterance) pairs: {len(per_utt)}")
print(f"Pairs appearing in 2+ strategies: {(per_utt >= 2).sum()} "
      f"({(per_utt >= 2).mean():.1%} of pairs)")
print(f"Pairs appearing in all 4 strategies: {(per_utt == 4).sum()}")

**Zero overlap on exact-match comparison.** No (seed, utterance) pair is produced by more than one strategy in the aggregate. The four strategies produce textually distinct variants as delivered to users — supporting the design claim that they are complementary rather than redundant.

**Note on how dedup interacts with this finding:** `evaluate.py`'s `deduplicate()` step removes exact-text duplicates across the combined variant pool before writing the CSV, using a first-seen-wins rule. This is intentional behavior — the tool's job is to deliver distinct variants to users, and keeping both copies of an identically-worded variant would waste space without adding information. Near-duplicates (e.g. "turn off the lights" vs. "turn the lights off") are preserved because they are *distinct variants* with different word orders and are exactly what the tool is designed to produce. The 0% overlap measured here is the user-facing behavior and what we wanted to test.

**What this doesn't tell us:** whether strategies would produce overlapping output absent dedup. That would be a different question — about prompt diversity at the generator level — and is reserved for item 3 on the testing checklist.

---

## Section 6: Known metric limitations

Beyond describing strategy behavior, the aggregate surfaces three specific cases where allo's evaluation metrics produce misleading scores. None of these are bugs — they are expected properties of the chosen models being exposed by running over a linguistically diverse seed set. Users thresholding on these metrics should know about them.

Each subsection below cites specific variants from the aggregate so users can see what the limitation looks like in practice, then concludes with a labeled hypothesis about what might address it in future work. Hypotheses are explicitly speculative and are offered as directions to investigate, not established improvements.

### Semantic similarity: `all-MiniLM-L6-v2`

The embedding model is sensitive to named entity substitutions in ways that overshoot intuitive notions of "same meaning." It is also weak at resolving deep synonymy, undershooting when two utterances express the same intent via different vocabulary.

In [ ]:
# MLM variants that dropped below sim=0.30 — overwhelmingly named-entity substitutions
mlm_low = df[(df['strategy_family'] == 'mlm_substitution') &
             (df['semantic_similarity'] < 0.30)].copy()
print(f"MLM variants with similarity < 0.30: {len(mlm_low)} "
      f"({len(mlm_low) / (df['strategy_family']=='mlm_substitution').sum():.1%} of MLM)")
print('\nAll of them (ordered by similarity):')
for _, row in mlm_low.sort_values('semantic_similarity').iterrows():
    print(f"  sim={row['semantic_similarity']:.3f}  '{row['seed']}' → '{row['utterance']}'")

**Pattern:** Single-word substitutions on seeds with named entities or content-carrying nouns produce the lowest similarity scores. "Babylon" vs. "Australia", "refund" vs. "clue", "highway" vs. "ambiguity" — the variants are surface-valid English and allo is not wrong to generate them, but the embedding correctly detects that the referent changed substantially. This is the metric behaving as intended; it's worth flagging because **users running MLM on seeds with named entities should expect a long low-similarity tail, and threshold accordingly.**

The complementary pattern in LLM paraphrase is the opposite problem — the variant preserves meaning but drops vocabulary overlap:

In [ ]:
# LLM paraphrase variants scoring below sim=0.30 — many are arguably semantically valid
llm_low = df[(df['strategy_family'] == 'llm_paraphrase') &
             (df['semantic_similarity'] < 0.30)].copy()
print(f"LLM paraphrase variants with similarity < 0.30: {len(llm_low)} "
      f"({len(llm_low) / (df['strategy_family']=='llm_paraphrase').sum():.1%} of LLM)")
print('\nTop 8:')
for _, row in llm_low.nsmallest(8, 'semantic_similarity').iterrows():
    print(f"  sim={row['semantic_similarity']:.3f}  '{row['seed']}' → '{row['utterance']}'")

**Pattern:** paraphrases that share little vocabulary with the seed score low, even when a human reader would call them clearly synonymous. "can you select a laid back track" from "play something uh relaxing" preserves intent but swaps almost every content word, and the embedding cannot recognize the synonymy.

**Hypothesis for future work (speculative):** A stronger sentence embedding model (e.g. `all-mpnet-base-v2`, larger and trained on more paraphrase data) might resolve synonymy better than `all-MiniLM-L6-v2`. The tradeoff is size and inference speed — MiniLM was chosen in part for lightweight local execution. Worth benchmarking on a held-out set of human-judged paraphrase pairs before any swap. This is a direction to investigate, not a claimed improvement.

### Perplexity: GPT-2

GPT-2 perplexity has two known weaknesses that the aggregate exposes. One is length-sensitivity — already documented in `evaluate.py`'s docstring. The other is a compound effect: filled-pause tokens in short utterances produce extreme perplexity values that don't reflect actual unnaturalness.

In [ ]:
# Short variants (≤3 words) with high similarity but catastrophic perplexity
short_sim_fine = df.copy()
short_sim_fine['n_words'] = short_sim_fine['utterance'].str.split().str.len()
short_sim_fine = short_sim_fine[
    (short_sim_fine['n_words'] <= 3) &
    (short_sim_fine['semantic_similarity'] > 0.85) &
    (short_sim_fine['perplexity'] > 5000)
]
print(f"Variants that are short (≤3 words), semantically close (sim > 0.85), ")
print(f"but score catastrophically bad perplexity (ppl > 5000): {len(short_sim_fine)}")
print(f"\nTop 6:")
for _, row in short_sim_fine.nlargest(6, 'perplexity').iterrows():
    print(f"  {int(row['n_words'])}w  sim={row['semantic_similarity']:.2f}  "
          f"ppl={row['perplexity']:>8.0f}  '{row['seed']}' → '{row['utterance']}'")

**Pattern:** Two-word and three-word utterances can score perplexity in the tens of thousands even when the similarity score confirms they are semantically near-identical to the seed. "call mom" (ppl=61,611, sim=0.94) from "call call mom" is the clearest example. The cause is mechanical: GPT-2 perplexity averages per-token negative log-likelihood, and very short sequences have almost no conditioning context, which inflates the per-token loss.

The second issue is specific to filled-pause tokens — `uh`, `um` — in seeds that contain them:

In [ ]:
# Compare variants that preserve vs drop the filled pause, per seed
import re

def has_filler(s):
    return bool(re.search(r'\b(uh|um)\b', s.lower()))

filler_seeds = df[df['seed'].apply(has_filler)]['seed'].unique()
print(f"Seeds containing filled pauses: {len(filler_seeds)}")
print()
for seed in filler_seeds:
    sub = df[df['seed'] == seed].copy()
    sub['preserves_filler'] = sub['utterance'].apply(has_filler)
    with_f = sub[sub['preserves_filler']]
    without_f = sub[~sub['preserves_filler']]
    if len(with_f) and len(without_f):
        ratio = with_f['perplexity'].median() / without_f['perplexity'].median()
        print(f"Seed: '{seed}'")
        print(f"  Preserves filler  (n={len(with_f):>3}): median ppl = {with_f['perplexity'].median():>8.1f}")
        print(f"  Drops filler      (n={len(without_f):>3}): median ppl = {without_f['perplexity'].median():>8.1f}")
        print(f"  Ratio: {ratio:.1f}×")
        print()

**Pattern:** When the seed contains a filled pause, variants that preserve the filler score higher perplexity than variants that drop it — but the effect size depends dramatically on utterance length.

- On **long seeds** with context around the filler ("um remind me to call the doctor tomorrow morning", "can you um turn off the lights..."), preserving the filler increases median perplexity by about 2×. Real effect, modest magnitude.
- On **short seeds** ("play something uh relaxing"), preserving the filler increases median perplexity by about 83×. This is what produced the extreme tail documented in Section 4.

The compound is what matters: perplexity only becomes pathological when the filled pause lacks sufficient surrounding context for GPT-2 to condition on. On longer seeds, the metric still penalizes filler preservation, but within a reasonable range.

**Hypothesis for future work (speculative):** Length-normalized perplexity (e.g. dividing by sqrt(n_tokens)) is a standard technique that would dampen the short-string pathology. A modern smaller model (e.g. a distilled GPT-2 replacement trained on more recent text) might also handle filler tokens better, since the GPT-2 training distribution is older and under-represents conversational disfluency. Both are directions to investigate; neither is being claimed as an improvement.

**Practical recommendation for users now:** applying `--filter-max-perplexity` aggressively is risky. At a threshold of 500, 11% of variants are removed, and those variants cluster around short seeds and disfluent seeds rather than around genuinely unnatural output. Users who do want to filter on perplexity should inspect what's being removed before committing to a threshold.

---

## Summary of findings

1. **Per-strategy similarity medians separate cleanly:** constrained 0.89, mlm_substitution 0.87, llm_paraphrase 0.83, expansion 0.64. The paraphrase-class strategies form a tight band; expansion sits well below by design.
2. **Per-strategy perplexity medians are all in the 99–154 range** — within GPT-2's typical natural-English band. Constrained produces the most fluent output (median 99); MLM and expansion the least (153, 154).
3. **Length affects both metrics consistently.** Similarity rises with length (short utterances get more disrupted by each change); perplexity falls with length (GPT-2 is more confident on longer sequences). Both effects are properties of the metrics, not the strategies.
4. **The README's 0.55 similarity threshold is defensible.** It sits in the valley between expansion and paraphrase-class distributions, removing 29% of expansion variants while preserving ~95–99% of the others. The 0.70 threshold (without expansion) is more aggressive than users may realise — it discards 17% of llm_paraphrase and 16% of mlm_substitution.
5. **The extreme perplexity tail is almost entirely explained by two seeds** (`call call mom` and `play something uh relaxing`). These produce 85% of ppl > 10,000 variants. The cause is GPT-2's known length-sensitivity on very short strings plus a compound effect with filled-pause tokens. Not a quality signal.
6. **Zero cross-strategy overlap in the aggregate.** The four strategies produce textually distinct variants as delivered to users, supporting the design claim that they are complementary.
7. **Three metric limitations surfaced by this study** (Section 6): MLM + named entities produces correctly-low similarity that users should expect; LLM paraphrase occasionally preserves meaning through vocabulary-drops that the embedding undershoots; GPT-2 perplexity is pathological on filled pauses in short seeds. Hypotheses for addressing each are flagged as speculative, not established.

**Implications for the README and docstrings:**

1. **Restore the "Interpreting scores across strategies" guidance** with the measured medians and IQRs from Section 1.
2. **Document length-sensitivity in both metrics.** Users applying similarity or perplexity filters should know that short seeds are disproportionately affected.
3. **Consider noting `0.65` as an alternative similarity floor with expansion.** It removes roughly half of expansion's output while preserving ~88% of paraphrase-class variants — a stricter setting for users who want to discard more of expansion's fringe.
4. **Add a "Known metric limitations" subsection to the README.** The three patterns in Section 6 are specific, reproducible, and affect how users should interpret scores. Worth documenting in a few sentences directly under "Evaluation metrics".
5. **Expand the `evaluate.py` perplexity docstring.** It already notes length-sensitivity; the filled-pause compound and the concrete "call mom" example are worth adding.

Next step: draft `evaluation/results/scoring_ranges/scoring_ranges.md` citing the plots, `scoring_summary.csv`, and Section 6's three limitations. Then apply the README and docstring updates from the implication list.